# Table of Contents
- 1. Neural Network and its applications, Single layer neural Network
- 2. Activation Functions: Sigmoid, Hyperbolic Tangent, ReLu
- 3. Overview of Back propagation of errors
- 4. Lab Assignment: Implement the different activation functions on the Dataset

### 1. Neural Network and its Applications, Single Layer Neural Network

Since you already know classical ML (linear/logistic regression, decision trees, gradient boosting) and data analytics workflows, the key conceptual shift here is **representation learning** and **composable differentiable computation graphs**:

- **Classical ML vs. Deep Neural Networks:** Tabular ML relies heavily on domain-engineered features and shallow decision boundaries. Neural networks learn hierarchical representations directly from raw or structured inputs by stacking linear projections ($z = Wx + b$) and non-linearities.
- **Where DNNs Shine vs. Where Tabular/Classical ML Wins:** Deep learning excels on unstructured/perceptual data (vision, audio, text, multimodal streams) and large datasets where capacity pays off. For medium-sized structured/tabular datasets, gradient-boosted trees (XGBoost, LightGBM, CatBoost) and regularized linear models often remain strong baselines and are faster to iterate on.
- **Single Layer Neural Network (Perceptron / Single-Layer Feedforward Unit):**
  - Takes input vector $x \in \mathbb{R}^d$, weights $w \in \mathbb{R}^d$, bias $b \in \mathbb{R}$, computes pre-activation $z = w^\top x + b$, and applies activation $\hat{y} = a = g(z)$.
  - **Classic Limitation (XOR / Linear Separability):** A single linear threshold or monotonic single neuron can only carve linear decision boundaries in input space—it cannot solve non-linearly separable problems like XOR without feature engineering or hidden layers/non-linearity.
  - **Edge Cases & Practical Pitfalls:**
    1. *Unscaled features:* Large-magnitude features skew weight updates and distance/dot-product scales.
    2. *Dead/zero initialization or symmetry:* Initializing all weights identically makes symmetric units update identically; zero weights stall gradients in linear/symmetric setups.
    3. *Numerical overflow/underflow:* Extreme values in $z$ can cause `exp(z)` overflow (`inf`/`nan`) or floating-point precision loss.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:


np.random.seed(42)

# Single-layer unit demo: z = w . x + b
def single_layer_forward(X, w, b, activation_fn):
    z = X @ w + b
    a = activation_fn(z)
    return z, a

X_sample = np.array([
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0],
])
w_demo = np.array([1.5, -2.0])
b_demo = 0.25

z_out, _ = single_layer_forward(X_sample, w_demo, b_demo, lambda z: z)
print("Pre-activations z:", np.round(z_out, 4))

# Edge case: overflow in naive np.exp for large positive/negative z
extreme_z = np.array([-1000.0, -50.0, 0.0, 50.0, 1000.0])
with np.errstate(over='ignore', invalid='ignore'):
    naive_sig = 1.0 / (1.0 + np.exp(-extreme_z))
print("Naive sigmoid on extreme z:", naive_sig)

## 2. Activation Functions: Sigmoid, Hyperbolic Tangent ($\tanh$), and ReLU

Why do we need non-linear activation functions? Composing purely linear layers ($W_2(W_1 x + b_1) + b_2 = (W_2 W_1)x + (W_2 b_1 + b_2)$) collapses an entire deep network into a **single linear transformation**, regardless of depth.

| Activation | Formula $\sigma(z)$ / $g(z)$ | Output Range | Derivative $g'(z)$ | Key Strengths & Edge Cases |
| :--- | :--- | :--- | :--- | :--- |
| **Sigmoid** | $\frac{1}{1 + e^{-z}}$ | $(0, 1)$ | $\sigma(z)(1 - \sigma(z))$ | Great for output probabilities in binary classification; **vanishing gradients** for $|z| \gg 0$ (max derivative is $0.25$ at $z=0$); output is **not zero-centered** ($>0$), causing zig-zag updates in subsequent layers; naive `exp(-z)` overflows for large negative $z$. |
| **Tanh** | $\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$ | $(-1, 1)$ | $1 - \tanh^2(z)$ | **Zero-centered** (often better than sigmoid in hidden layers); max derivative is $1.0$ at $z=0$, but still **saturates (vanishes)** for $|z| \gg 0$; naive `exp(z) - exp(-z)` can overflow for large $|z|$ (prefer `np.tanh`). |
| **ReLU** | $\max(0, z)$ | $[0, \infty)$ | $1$ if $z > 0$ else $0$ (at $z=0$, subgradient in $[0,1]$) | Fast, sparse activations, avoids saturation for $z > 0$; **Dying ReLU edge case** where a unit always outputs $0$ across the dataset (zero gradient, never recovers with standard SGD); **unbounded output** can lead to large activations/exploding values if unregularized/unscaled. |

### Numerically Stable Sigmoid Edge Case
- For $z \ge 0$: $\sigma(z) = \frac{1}{1 + e^{-z}}$ (avoids large positive exponent).
- For $z < 0$: $\sigma(z) = \frac{e^{z}}{1 + e^{z}}$ (avoids `exp(-z)` overflow when $z = -1000$).

In [ ]:

def sigmoid(z):
    z = np.asarray(z, dtype=np.float64)
    out = np.empty_like(z, dtype=np.float64)
    pos_mask = z >= 0
    neg_mask = ~pos_mask
    out[pos_mask] = 1.0 / (1.0 + np.exp(-z[pos_mask]))
    exp_z = np.exp(z[neg_mask])
    out[neg_mask] = exp_z / (1.0 + exp_z)
    return out


def sigmoid_deriv(z):
    s = sigmoid(z)
    return s * (1.0 - s)


def tanh_act(z):
    return np.tanh(z)


def tanh_deriv(z):
    t = np.tanh(z)
    return 1.0 - t ** 2


def relu(z):
    return np.maximum(0.0, z)


def relu_deriv(z):
    return (np.asarray(z) > 0.0).astype(np.float64)


extreme_z = np.array([-1000.0, -50.0, 0.0, 50.0, 1000.0], dtype=np.float64)  # <-- NEW

# Verify stability on extreme values
print("Stable sigmoid on extreme_z:", sigmoid(extreme_z))

# Plot activations and derivatives
z_grid = np.linspace(-6, 6, 400)  # will generate 400 points between -6 and 6


sample_z = np.array([-4.0, -1.0, 0.0, 1.0, 4.0])  # <-- NEW
print("\n=== Sample Outputs Before Plotting ===")  # <-- NEW
print("Sample z:        ", sample_z)  # <-- NEW
print("Sigmoid(z):      ", np.round(sigmoid(sample_z), 4))  # <-- NEW
print("Sigmoid'(z):     ", np.round(sigmoid_deriv(sample_z), 4))  # <-- NEW
print("Tanh(z):         ", np.round(tanh_act(sample_z), 4))  # <-- NEW
print("Tanh'(z):        ", np.round(tanh_deriv(sample_z), 4))  # <-- NEW
print("ReLU(z):         ", np.round(relu(sample_z), 4))  # <-- NEW
print("ReLU'(z):        ", np.round(relu_deriv(sample_z), 4))  # <-- NEW
print("z_grid min/max:  ", z_grid.min(), "to", z_grid.max(), f"({len(z_grid)} pts)")  # <-- NEW

# visualizing activations and derivatives
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(z_grid, sigmoid(z_grid), label="Sigmoid")
axes[0].plot(z_grid, tanh_act(z_grid), label="Tanh")
axes[0].plot(z_grid, relu(z_grid), label="ReLU")
axes[0].set_title("Activation Functions g(z)")
axes[0].set_xlabel("z")
axes[0].set_ylabel("g(z)")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(z_grid, sigmoid_deriv(z_grid), label="Sigmoid'")
axes[1].plot(z_grid, tanh_deriv(z_grid), label="Tanh'")
axes[1].plot(z_grid, relu_deriv(z_grid), label="ReLU'")
axes[1].set_title("Derivatives g'(z)")
axes[1].set_xlabel("z")
axes[1].set_ylabel("g'(z)")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.datasets import make_classification

# 1. Generate a sample dataset with 200 rows and 4 features
X_data, y_data = make_classification(
    n_samples=200, # number of samples in the dataset
    n_features=4, # number of features in the dataset
    n_informative=3, # number of informative features : usefull features
    n_redundant=1, # number of redundant features : repeated features
    random_state=42,
)

# print("X_data shape:", X_data.shape)
# print("y_data shape:", y_data.shape)
# print("_" * 60)
# # display firt 5 rows of the dataset
# print("First 5 rows of X_data:\n", X_data[:5])
# print("First 5 rows of y_data:\n", y_data[:5])
# print("_" * 60)

# Inject an extreme-scale edge case on the last 5 rows to test numerical behavior
X_edge = X_data.copy()
X_edge[-5:, 0] *= 100.0 # scale the first feature of the last 5 rows by 100 to create extreme values

# Initialize weights w (shape (4,)) and bias b (scalar)
rng = np.random.default_rng(42)
w_init = rng.normal(loc=0.0, scale=0.5, size=X_data.shape[1])
b_init = 0.0

# Compute linear pre-activations z for X_data and X_edge
z_clean = X_data @ w_init + b_init
z_extreme = X_edge @ w_init + b_init

# Apply sigmoid, tanh_act, and relu on z_clean and z_extreme
activations = {
    "Sigmoid": (sigmoid, lambda a: np.mean((a <= 1e-4) | (a >= 1.0 - 1e-4))), # 
    "Tanh": (tanh_act, lambda a: np.mean(np.abs(a) >= 0.99)),
    "ReLU": (relu, lambda a: np.mean(a <= 0.0)),
}


for name, (fn, sat_or_zero_fn) in activations.items():
    a_clean = fn(z_clean)
    a_ext = fn(z_extreme)
    print(f"=== {name} ===")
    print(
        f"  Clean   -> mean: {a_clean.mean():.4f}, std: {a_clean.std():.4f}, "
        f"min: {a_clean.min():.4f}, max: {a_clean.max():.4f}, "
        f"sat/zero rate: {sat_or_zero_fn(a_clean):.2%}"
    )
    print(
        f"  Extreme -> mean: {a_ext.mean():.4f}, std: {a_ext.std():.4f}, "
        f"min: {a_ext.min():.4f}, max: {a_ext.max():.4f}, "
        f"sat/zero rate: {sat_or_zero_fn(a_ext):.2%}"
    )

    # Plot histograms comparing clean vs. extreme-scale activations
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, (fn, _)) in zip(axes, activations.items()):
    a_clean = fn(z_clean)
    a_ext = fn(z_extreme)
    ax.hist(a_clean, bins=25, alpha=0.7, label="Clean (200 rows)")
    ax.hist(a_ext[-5:], bins=10, alpha=0.85, label="Extreme rows (last 5)")
    ax.set_title(f"{name} Activations")
    ax.set_xlabel("Activation output")
    ax.set_ylabel("Count")
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()